**IMPORTAÇÕES NECESSARIAS**

*   **IMPORT OS :**   Gerencia pastas e caminhos de arquivos no sistema.

*   **IMPORT REQUESTS :**   Faz o download das páginas e arquivos da internet.
*   **IMPORT PANDAS AS PD :**   Processa as planilhas, filtra dados e cria tabelas.
*   **IMPORT JSON :**   Salva e lê a memória (cache) para o código ser rápido.
*   **IMPORT WARNINGS :**   Esconde avisos de formatação irrelevantes do Excel. (Somente para melhor visualização)
*   **IMPORT URLLIB3 :**   Ignora alertas de segurança (SSL) de sites oficiais. (Somente para melhor visualização)
*   **FROM BS4 IMPORT BEAUTIFULSOUP :**   Varre o site para encontrar links de download.
*   **FROM URLLIB.PARSE IMPORT URLJOIN :**   Une partes de links para formar a URL final.
*   **FROM GOOGLE.COLAB IMPORT DRIVE :**   Conecta o script às pastas do seu Google Drive.
*   **FROM IO IMPORT BYTESIO :**   Trata dados baixados como se fossem um arquivo físico.
*   **FROM DATETIME IMPORT DATETIME :**   Identifica a data atual.

In [36]:
import os
import requests
import pandas as pd
import json
import warnings
import urllib3
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from google.colab import drive
from io import BytesIO
from datetime import datetime

**ABRINDO A "MESA DE TRABALHO"**



1.   Abre o google drive para iniciar os "trabalhos" pedindo permissão para acessa-lo.

2.   Silencia os avisos de que o arquivo que esta abrindo esta inconpleto ou digitado errado.




In [37]:
# Configurações do Google colab
drive.mount('/content/drive', force_remount=True)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

Mounted at /content/drive


**Informações "Traduzidas" para facilitar a leitura e entendimento do codigo**



*   GRUPOS_VEICULOS = Grupo de veiculos para fazer a leitura e futura soma.
*   MESES_CONVERTIDO_EM_NUMEROS / MESES_NOME = Ensinando que cada mês tem um numero.



In [38]:
GRUPOS_VEICULOS = {
    'Motocicletas': ['CICLOMOTOR', 'MOTONETA', 'MOTOCICLETA', 'SIDE-CAR', 'TRICICLO', 'QUADRICICLO'],
    'Automóveis': ['AUTOMOVEL', 'CAMIONETA', 'CAMINHONETE', 'UTILITARIO'],
    'Ônibus': ['ONIBUS', 'MICRO-ONIBUS'],
    'Caminhões': ['CAMINHAO', 'CAMINHAO TRATOR'],
    'Outros': ['BONDE', 'CHASSI PLATAF', 'REBOQUE', 'SEMI-REBOQUE', 'OUTROS', 'TRATOR ESTEI', 'TRATOR RODAS']
}

MESES_CONVERTIDO_EM_NUMEROS = {
    'janeiro': 1, 'fevereiro': 2, 'março': 3, 'abril': 4, 'maio': 5, 'junho': 6,
    'julho': 7, 'agosto': 8, 'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12
}
MESES_NOME = {v: k.capitalize() for k, v in MESES_CONVERTIDO_EM_NUMEROS.items()}


**Guardando dentro de uma "mochila" os dados**
1.   Define onde o arquivo ficara, o local que fica armazenado.

2.   Define um arquivo de arquivo rapido.

3.   Se a pasta não existe ele cria uma.

**Inicio das Funções e movimento de dados**

*   carregar_cache: Ela serve para verificar se tem o arquivo solicitado no arquivo rapido.
*   Salvar_arquivos_no_json: Esse vem se caso o de cima não for usado, ele serve para salvar o arquivo dentro do arquivo rapido.
*   baixar_se_nao_existir: Se caso as duas de cima não forem usados essa vai procurar o arquivo em um site externo utilizando a biblioteca BEAUTIFULSOUP.



In [39]:

CAMINHO_PARA_O_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Frota de Veiculos/'
ARQUIVO_INFO_RAPIDAS = os.path.join(CAMINHO_PARA_O_DRIVE, 'Arquivo_para_info_rapidas.json')

if not os.path.exists(CAMINHO_PARA_O_DRIVE):
    os.makedirs(CAMINHO_PARA_O_DRIVE)

def carregar_cache():
    if os.path.exists(ARQUIVO_INFO_RAPIDAS):
        try:
            with open(ARQUIVO_INFO_RAPIDAS, 'r') as f: return json.load(f)
        except: return {}
    return {}

def Salvar_arquivos_no_json(dados):
    with open(ARQUIVO_INFO_RAPIDAS, 'w') as f: json.dump(dados, f)

def baixar_se_nao_existir(ano, mes_nome):
    mes_num = MESES_CONVERTIDO_EM_NUMEROS[mes_nome.lower()]
    Nome_do_arquivo_redefinido = f"{ano}_{mes_num}.xlsx"
    caminho = os.path.join(CAMINHO_PARA_O_DRIVE, Nome_do_arquivo_redefinido)

    if os.path.exists(caminho):
        return caminho

    url_p = f"https://www.gov.br/transportes/pt-br/assuntos/transito/conteudo-Senatran/frota-de-veiculos-{ano}"
    try:
        res = requests.get(url_p, verify=False, timeout=15)
        sopa = BeautifulSoup(res.text, 'html.parser')
        ancora = None
        for tag in sopa.find_all(['p', 'strong', 'h2']):
            txt = tag.get_text().lower()
            if "frota nacional" in txt and mes_nome.lower() in txt:
                ancora = tag
                break

        if ancora:
            links = [l for l in ancora.find_all_next('a', limit=15) if '.xls' in l.get('href', '').lower()]
            if len(links) >= 2:
                url_dl = urljoin(url_p, links[1]['href'])
                conteudo = requests.get(url_dl, verify=False).content
                with open(caminho, 'wb') as f: f.write(conteudo)
                return caminho
    except: pass
    return None



**Centro das operações**

A função processar_frota atual como o cérebro estratégico do sistema. É ela quem gerencia a interação com o usuário e define o escopo do trabalho.


*   **Interação Crucial:** Solicita ao usuário o ano e o mês de referência.
*   **Filtro Inteligente:** Refina as intenções do usuário para preparar o processamento em lote (loop).
*   **Eficiência Operacional:** Garante que o sistema não desperdice tempo ou recursos tentando baixar meses inexistentes ou que não foram solicitados.



In [40]:
def processar_frota():
    memoria_rapida = carregar_cache()
    dia_hora_atual = datetime.now()

    pergunta_ano = input("Digite o ano (ex: 2026): ").strip()
    pergunta_mes = input("Digite o mês (Vazio para tudo): ").strip()
    ano_escolhino = int(pergunta_ano)

    if pergunta_mes:
        meses_para_usar = [MESES_NOME[int(pergunta_mes)]] if pergunta_mes.isdigit() else [pergunta_mes.capitalize()]
    else:
        limite = dia_hora_atual.month - 1 if ano_escolhino == dia_hora_atual.year else 12
        meses_para_usar = [MESES_NOME[m] for m in range(1, limite + 1)]

    dados_finais = []

    for nome_mes in meses_para_usar:
        id_arquivo = f"{ano_escolhino}_{MESES_CONVERTIDO_EM_NUMEROS[nome_mes.lower()]}.xlsx"
        resumo = memoria_rapida.get(id_arquivo)

        if not resumo:
            caminho = baixar_se_nao_existir(ano_escolhino, nome_mes)
            if caminho:
                try:
                    df = pd.read_excel(caminho, header=2)
                    mascara = df.iloc[:, 1].astype(str).str.strip().str.upper().isin(['SAO PAULO', 'SÃO PAULO'])
                    df_sp = df[mascara].head(1).copy()

                    if not df_sp.empty:
                        df_sp.columns = [str(c).strip().upper() for c in df_sp.columns]
                        resumo = {'Ano': ano_escolhino, 'Mês': nome_mes, 'Município': 'SÃO PAULO'}
                        total_linha = 0

                        for grupo, colunas in GRUPOS_VEICULOS.items():
                            c_existentes = [c for c in colunas if c in df_sp.columns]
                            soma = int(sum([pd.to_numeric(df_sp[c], errors='coerce').fillna(0).iloc[0] for c in c_existentes]))
                            resumo[grupo] = soma
                            total_linha += soma

                        resumo['Frota Total'] = total_linha
                        memoria_rapida[id_arquivo] = resumo
                        Salvar_arquivos_no_json(memoria_rapida)
                except: continue

        if resumo:
            dados_finais.append(resumo)

    if dados_finais:
        tabela_final_apresentar = pd.DataFrame(dados_finais)
        COLUNAS_DA_TABELA = ['Ano', 'Mês', 'Município', 'Motocicletas', 'Automóveis', 'Ônibus', 'Caminhões', 'Outros', 'Frota Total']
        print(f"\nRELATÓRIO DE FROTA ({ano_escolhino})")
        display(tabela_final_apresentar[COLUNAS_DA_TABELA].style.hide(axis='index'))
    else:
        print(f"Dados de {ano_escolhino} ainda não disponíveis ou não encontrados.")

processar_frota()

Digite o ano (ex: 2026): 2025
Digite o mês (Vazio para tudo): 

RELATÓRIO DE FROTA (2025)


Ano,Mês,Município,Motocicletas,Automóveis,Ônibus,Caminhões,Outros,Frota Total
2025,Janeiro,SÃO PAULO,1571259,7789356,93729,185942,124722,9765008
2025,Fevereiro,SÃO PAULO,1577400,7807572,93796,186032,125203,9790003
2025,Março,SÃO PAULO,1584881,7821537,94048,186274,125608,9812348
2025,Abril,SÃO PAULO,1593955,7839056,94384,186848,125896,9840139
2025,Maio,SÃO PAULO,1602752,7852525,94410,187106,126256,9863049
2025,Junho,SÃO PAULO,1610563,7863187,94513,187300,126512,9882075
2025,Julho,SÃO PAULO,1610563,7863187,94513,187300,126512,9882075
2025,Agosto,SÃO PAULO,1623995,7850876,93872,186412,163542,9918697
2025,Setembro,SÃO PAULO,1631009,7859550,93964,186665,164003,9935191
2025,Outubro,SÃO PAULO,1637958,7868255,94077,187040,164386,9951716
